# AdMul VUA All — checkpoint đầy đủ mỗi epoch

## Lần đầu
1. Import notebook này. Bật **GPU T4** và **Internet**. Chỉ dùng **một GPU**, FP32.
2. **Add Input** Dataset chứa `admul_data.zip` (bản ZIP dữ liệu gốc), hoặc dữ liệu đã giải nén.
   Có thể gắn Output `lminhhinlminhhin/full-data` để lấy dữ liệu; checkpoint cũ sẽ không được nạp.
3. Giữ `MODE = 'auto'`, seed 4, tối đa 3 epoch/phiên. **Save Version → Save & Run All**.

## Các phiên sau
1. Gỡ Input cũ, gắn **Output phiên gần nhất của notebook này** (chỉ một bản).
2. Giữ nguyên notebook và `MODE = 'auto'`, rồi **Save & Run All**.
3. Tự khôi phục từ epoch đã hoàn thành: dự kiến **1–3 → 4–6 → 7–8**.
   Nếu thiếu checkpoint khi bạn đã chạy trước đó, đặt `MODE = 'resume'` để bắt buộc báo lỗi,
   tránh vô tình bắt đầu lại. Không tự khôi phục được nếu bạn không gắn Output phiên trước.

## Được giữ lại trong Output
- `AdMulPaper/run/epochs/epoch_01.pt`, `epoch_02.pt`, …: **tất cả checkpoint epoch**,
  gồm model, AdamW, scheduler, RNG Python/NumPy/PyTorch/CUDA, bước train, lịch sử và epoch tốt nhất.
- `data/data/`: dữ liệu gốc; `run/features.pt`: dữ liệu tokenize, gồm thứ tự BSD đã shuffle,
  lưu một lần, không lặp lại dữ liệu lớn trong mỗi checkpoint.
- `run/identity.json`: SHA-256 dữ liệu, code, features và pretrained model; phiên bản môi trường.
  Resume từ chối khi đầu vào, cấu hình, môi trường hoặc GPU khác.
- `run/history.json`, `result.json`, `events.jsonl`, `console.log`, tokenizer, pretrained model và code.
  Model tốt nhất được chỉ ra bởi `best_checkpoint` trong `result.json`.

## Sát paper và code gốc
Giữ nguyên model, processor, sampler, train/val và cấu hình VUA All từ repo tác giả.
Transformers **4.18.0**, NumPy **1.21.3**, **Transformers AdamW gốc**, 8 epoch, batch 64,
LR 3e-5, warmup 2 epoch, max length 150, seed 4, dropout 0, loss và lịch adversarial gốc.
Giữ toàn bộ VUA train/dev/test. BSD lấy số dòng bằng VUA train và shuffle sau khi cắt,
đúng `main_vua.py`; không tự dùng thêm SemCor dư hay thay cách tokenize/cắt câu.
Giữ ba lượt đánh giá mỗi epoch như tác giả; chỉ dùng **F1 dev** để chọn model.

**Khác biệt được ghi rõ:** bật gradient checkpointing để giảm activation, giữ RNG khi tính lại;
không dùng AMP, gradient accumulation hay multi-GPU. Dùng Python 3.10.14/PyTorch 1.13.1 CUDA 11.6
vì repo không công bố phiên bản Python/PyTorch. Pretrained DeBERTa-v3-base được khóa revision;
paper ghi GPU RTX 3090/CUDA 11.6; Kaggle T4 khác GPU và paper không cung cấp đầy đủ
trạng thái môi trường để bảo đảm trùng điểm tuyệt đối.
Các cờ deterministic được bật (warn-only cho phép toán chưa hỗ trợ); GPU vẫn có thể có sai khác số học.
Không bảo đảm điểm số y hệt paper; mục tiêu là giữ thuật toán và khôi phục trạng thái đầy đủ.

Notebook cài môi trường Python riêng tại `/tmp` để chạy thư viện cũ trên Kaggle mới.
Nếu cài đặt hoặc CUDA không tương thích, sẽ báo lỗi; không tự đổi sang thư viện mới.
Giới hạn phiên 10,5 giờ tính từ cell đầu, tối đa 3 epoch; kiểm tra thời gian trước mỗi epoch
và giữa các batch. Khi hết ngân sách, thoát bình thường để giữ Output; epoch dở sẽ chạy lại
từ checkpoint epoch hoàn chỉnh trước đó. Nếu Kaggle cưỡng bức dừng trước khi lưu Output,
không thể bảo đảm lấy lại file, nên dùng Save & Run All và giữ khoảng đệm thời gian.

Mỗi full checkpoint khoảng vài GB; giữ 8 epoch cần nhiều dung lượng. Không tạo ZIP lặp checkpoint.
Chỉ nạp các file `.pt` do chính bạn tạo. File chứa cả dữ liệu pickle của optimizer và RNG.

Nguồn: [paper](https://aclanthology.org/2023.findings-acl.96/),
[repo tác giả](https://github.com/SilasTHU/AdMul),
[Transformers 4.18 optimizer](https://github.com/huggingface/transformers/blob/v4.18.0/src/transformers/optimization.py).


In [ ]:
# Đọc lại cell này ở mỗi phiên. Không cần đổi MODE nếu đã gắn đúng Output gần nhất.
import os, sys, time, json, shutil, subprocess, zipfile
from pathlib import Path
SESSION_STARTED = time.monotonic()
MODE = 'auto'   # auto: nối khi có checkpoint của notebook này, lần đầu chạy từ epoch 1.
                # resume: bắt buộc có checkpoint; fresh: bắt đầu thí nghiệm mới.
SEED = 4
MAX_SESSION_EPOCHS = 3
SESSION_HOURS = 10.5
INPUT_ROOT = Path('/kaggle/input')
WORK = Path('/kaggle/working/AdMulPaper')
assert MODE in ('auto', 'resume', 'fresh')
assert not WORK.exists(), 'Working directory đã tồn tại. Khởi động phiên Kaggle mới để tránh trộn kết quả.'
WORK.mkdir(parents=True)
markers = list(INPUT_ROOT.rglob('admul_run.json'))
if MODE == 'fresh':
    markers = []
if len(markers) > 1:
    raise ValueError(f'Có nhiều bản Output: {markers}. Chỉ gắn phiên gần nhất.')
if MODE == 'resume' and not markers:
    raise FileNotFoundError('Gắn Output phiên gần nhất có admul_run.json trước khi resume')
if markers:
    previous = markers[0].parent
    print('Restore workspace:', previous)
    shutil.copytree(previous, WORK, dirs_exist_ok=True)
    if MODE == 'resume' and not list((WORK / 'run/epochs').glob('epoch_*.pt')):
        raise FileNotFoundError('Output này chưa có epoch hoàn chỉnh để resume')
else:
    print('FRESH RUN: pretrained backbone only; no previous trained checkpoint')


In [ ]:
# Khôi phục từ ZIP gốc hoặc thư mục dữ liệu, bỏ metadata macOS.
if not (WORK / 'data/data/VUA_All/train.tsv').exists():
    archives = list(INPUT_ROOT.rglob('admul_data.zip'))
    if len(archives) > 1:
        raise ValueError('Chỉ gắn một bản admul_data.zip')
    if archives:
        with zipfile.ZipFile(archives[0]) as archive:
            for item in archive.infolist():
                parts = Path(item.filename).parts
                if (item.is_dir() or not parts or parts[0] != 'data'
                        or item.filename.startswith('__MACOSX') or parts[-1] == '.DS_Store'):
                    continue
                destination = (WORK / 'data').joinpath(*parts).resolve()
                if not destination.is_relative_to(WORK.resolve()):
                    raise ValueError('Unsafe archive path')
                destination.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(item) as source, destination.open('wb') as target:
                    shutil.copyfileobj(source, target)
    else:
        candidates = [p.parent.parent for p in INPUT_ROOT.rglob('train.tsv')
                      if p.parent.name == 'VUA_All'
                      and (p.parent.parent / 'semcor/threshold/train_3.tsv').exists()]
        if len(candidates) != 1:
            raise FileNotFoundError(f'Gắn đúng một Dataset dữ liệu gốc; tìm thấy {candidates}')
        shutil.copytree(candidates[0], WORK / 'data/data')
assert (WORK / 'data/data/VUA_All/train.tsv').exists(), 'Thiếu VUA All'
assert (WORK / 'data/data/semcor/threshold/train_3.tsv').exists(), 'Thiếu SemCor'


In [ ]:
SOURCES = {'LICENSE': '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'model.py': '# Author: Silas Cheung (Shenglong Zhang)\nimport torch\nimport torch.nn as nn\nimport argparse\nfrom transformers import AutoModel, AutoConfig\n\n\n# Gradient Reversal Layer\nclass GRLayer(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, x, lmbd=0.01):\n        ctx.lmbd = torch.tensor(lmbd)\n        return x.reshape_as(x)\n\n    @staticmethod\n    # 输入为forward输出的梯度\n    def backward(ctx, grad_output):\n        grad_input = grad_output.clone()\n        return ctx.lmbd * grad_input.neg(), None\n\n\n# Copied from transformers.models.bert.modeling_bert.BertPooler\n# DeBERTa does not have poolerout, so we implement manually\nclass Pooler(nn.Module):\n    def __init__(self, config):\n        super().__init__()\n        self.config = config\n        self.dense = nn.Linear(config.hidden_size, config.hidden_size)\n        self.activation = nn.Tanh()\n        self._init_weights(self.dense)\n\n    def _init_weights(self, module):\n        """Initialize the weights"""\n        if isinstance(module, nn.Linear):\n            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)\n            if module.bias is not None:\n                module.bias.data.zero_()\n\n    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:\n        # We "pool" the model by simply taking the hidden state corresponding\n        # to the first token.\n        first_token_tensor = hidden_states[:, 0]\n        pooled_output = self.dense(first_token_tensor)\n        pooled_output = self.activation(pooled_output)\n        return pooled_output\n\n\nclass Base_model(nn.Module):\n    def __init__(self, args):\n        super(Base_model, self).__init__()\n        self.plm = AutoModel.from_pretrained(args.DATA.plm)\n        for param in self.plm.parameters():\n            param.requires_grad = True\n\n    def _init_weights(self, module):\n        """Initialize the weights"""\n        if isinstance(module, (nn.Linear, nn.Embedding)):\n            module.weight.data.normal_(mean=0.0, std=self.plm.config.initializer_range)\n        elif isinstance(module, nn.LayerNorm):\n            module.bias.data.zero_()\n            module.weight.data.fill_(1.0)\n        if isinstance(module, nn.Linear) and module.bias is not None:\n            module.bias.data.zero_()\n\n\nclass BERT_base(Base_model):\n    def __init__(self, args):\n        super(BERT_base, self).__init__(args)\n        self.config = AutoConfig.from_pretrained(args.DATA.plm)\n        self.dropout = nn.Dropout(args.MODEL.dropout)\n        self.wsd_fc = nn.Linear(args.MODEL.embed_dim, args.MODEL.num_classes)\n        self.md_fc = nn.Linear(args.MODEL.embed_dim, args.MODEL.num_classes)\n        self._init_weights(self.wsd_fc)\n        self._init_weights(self.md_fc)\n\n    def forward(self, input_ids, type_ids, att_mask, domains):\n        embeddings = self.plm(input_ids, token_type_ids=type_ids, attention_mask=att_mask)\n        embeddings = embeddings.pooler_output\n        embeddings = self.dropout(embeddings)\n\n        wsd_indices = (domains == 0).nonzero(as_tuple=True)[0]\n        md_indices = (domains == 1).nonzero(as_tuple=True)[0]\n        wsd_embeddings = torch.index_select(embeddings, 0, wsd_indices)\n        md_embeddings = torch.index_select(embeddings, 0, md_indices)\n\n        wsd_out = self.wsd_fc(wsd_embeddings)\n        md_out = self.md_fc(md_embeddings)\n\n        return wsd_out, md_out\n\n\nclass DeBERTa_base(Base_model):\n    def __init__(self, args):\n        super(DeBERTa_base, self).__init__(args)\n        self.dropout = nn.Dropout(args.MODEL.dropout)\n        self.config = AutoConfig.from_pretrained(args.DATA.plm)\n        self.pooler = Pooler(config=self.config)\n        self.wsd_fc = nn.Linear(args.MODEL.embed_dim, args.MODEL.num_classes)\n        self.md_fc = nn.Linear(args.MODEL.embed_dim, args.MODEL.num_classes)\n        self.task_fc = nn.Linear(args.MODEL.embed_dim, args.MODEL.num_tasks)\n        self.label_fc0 = nn.Linear(args.MODEL.embed_dim, args.MODEL.num_classes)\n        self.label_fc1 = nn.Linear(args.MODEL.embed_dim, args.MODEL.num_classes)\n        self._init_weights(self.wsd_fc)\n        self._init_weights(self.md_fc)\n        self._init_weights(self.task_fc)\n        self._init_weights(self.label_fc0)\n        self._init_weights(self.label_fc1)\n\n    def forward(self, input_ids, type_ids, att_mask, adv_lmbd=0.01, domain_idx=1):\n        if self.training:\n            embeddings = self.plm(input_ids, token_type_ids=type_ids, attention_mask=att_mask)\n            embeddings = embeddings.last_hidden_state\n            pooler_embeddings = self.pooler(embeddings)\n            mean_embeddings = torch.mean(embeddings, dim=1)\n\n            half_batch_size = int(0.5 * pooler_embeddings.shape[0])\n            wsd_embeddings = pooler_embeddings[:half_batch_size]\n            md_embeddings = pooler_embeddings[half_batch_size:]\n\n            wsd_out = self.wsd_fc(wsd_embeddings)\n            md_out = self.md_fc(md_embeddings)\n\n            # marginal distribution: global out\n            mean_reverse = GRLayer.apply(mean_embeddings, adv_lmbd)\n            global_out = self.task_fc(mean_reverse)\n\n            # conditional distribution: local out\n            wsd_reverse = mean_reverse[:half_batch_size]\n            md_reverse = mean_reverse[half_batch_size:]\n            wsd_prob = torch.softmax(wsd_out, dim=1)\n            md_prob = torch.softmax(md_out, dim=1)\n\n            # align class 0\n            wsd_p0 = wsd_prob[:, 0].reshape(wsd_reverse.shape[0], 1)\n            wsd_feature0 = wsd_p0 * wsd_reverse\n            md_p0 = md_prob[:, 0].reshape(md_reverse.shape[0], 1)\n            md_feature0 = md_p0 * md_reverse\n            wsd_local_out0 = self.label_fc0(wsd_feature0)\n            md_local_out0 = self.label_fc0(md_feature0)\n\n            # align class 1\n            wsd_p1 = wsd_prob[:, 1].reshape(wsd_reverse.shape[0], 1)\n            wsd_feature1 = wsd_p1 * wsd_reverse\n            md_p1 = md_prob[:, 1].reshape(md_reverse.shape[0], 1)\n            md_feature1 = md_p1 * md_reverse\n            wsd_local_out1 = self.label_fc1(wsd_feature1)\n            md_local_out1 = self.label_fc1(md_feature1)\n\n            wsd_local_out = [wsd_local_out0, wsd_local_out1]\n            md_local_out = [md_local_out0, md_local_out1]\n\n            return wsd_out, md_out, global_out, wsd_local_out, md_local_out\n\n        else:\n            embeddings = self.plm(input_ids, token_type_ids=type_ids, attention_mask=att_mask)\n            embeddings = embeddings.last_hidden_state\n            pooler_embeddings = self.pooler(embeddings)\n\n            if domain_idx == 0:\n                out = self.wsd_fc(pooler_embeddings)\n            else:\n                out = self.md_fc(pooler_embeddings)\n\n            return out\n\n\nif __name__ == "__main__":\n    pass', 'data_process.py': 'import csv\nfrom transformers import AutoTokenizer\nimport os\n\n\n# parent class for dataset processor\nclass Data_Processor:\n    def __init__(self, args):\n        self.tokenizer = AutoTokenizer.from_pretrained(args.DATA.plm)\n        self.data_dir = args.DATA.data_dir\n        self.max_len = args.DATA.max_len  # the previously set max length of the input\n        self.plm = args.DATA.plm\n\n    def __str__(self):\n        pattern = \'\'\'Data Configs: \n        data_dir: {} \n        max_len: {}\'\'\'\n        return pattern.format(self.data_dir, self.max_len)\n\n    def _get_pair_examples(self, file_dir):\n        examples = []\n        with open(file_dir, \'r\', encoding=\'utf-8\') as f:\n            lines = csv.reader(f, delimiter=\'\\t\')\n            next(lines)  # skip the headline\n            for i, line in enumerate(lines):\n                if \'semcor\' in file_dir:\n                    sent_id, target_id, target_loc, lemma, pos, sentence, label = line\n                    domain = 0\n                else:\n                    target_loc, lemma, pos, sentence, label = line\n                    domain = 1\n\n                label = int(label)\n                inputs = self.tokenizer(lemma, sentence)\n                input_ids = inputs[\'input_ids\']\n                att_mask = inputs[\'attention_mask\']\n                if \'roberta\' in self.plm:\n                    len_input1 = len(self.tokenizer.tokenize(lemma, add_special_tokens=True))\n                    len_input2 = len(input_ids) - len_input1\n                    type_ids = len_input1 * [0] + len_input2 * [1]\n                else:\n                    type_ids = inputs[\'token_type_ids\']\n\n                cur_len = len(input_ids)\n                if cur_len < self.max_len:\n                    input_ids = input_ids + [self.tokenizer.pad_token_id] * (self.max_len - cur_len)\n                    type_ids = type_ids + [0] * (self.max_len - cur_len)\n                    att_mask = att_mask + [0] * (self.max_len - cur_len)\n                else:\n                    input_ids = input_ids[:self.max_len]\n                    type_ids = type_ids[:self.max_len]\n                    att_mask = att_mask[:self.max_len]\n\n                examples.append([input_ids, type_ids, att_mask, label, domain])\n                if (i + 1) % 10000 == 0:\n                    print(f\'{i + 1} sentences have been processed.\')\n\n            print(f\'{file_dir} finished.\')\n        return examples\n\n\nclass WSD_Processor(Data_Processor):\n    def __init__(self, args):\n        super(WSD_Processor, self).__init__(args)\n        self.threshold = args.DATA.wsd_threshold\n\n    # use toy data for testing\n    def get_wsd_train(self):\n        data_path = os.path.join(self.data_dir, f\'semcor/threshold/train_{self.threshold}.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n    def get_wsd_test(self):\n        data_path = os.path.join(self.data_dir, f\'semcor/threshold/test_{self.threshold}.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n    def get_wsd_toy(self):\n        data_path = os.path.join(self.data_dir, f\'semcor/toy.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n\nclass MD_Processor(Data_Processor):\n    def __init__(self, args):\n        super(MD_Processor, self).__init__(args)\n\n    def get_all_train(self):\n        data_path = os.path.join(self.data_dir, f\'VUA_All/train.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n    def get_all_val(self):\n        data_path = os.path.join(self.data_dir, f\'VUA_All/val.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n    def get_all_test(self):\n        data_path = os.path.join(self.data_dir, f\'VUA_All/test.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n    def get_verb_train(self):\n        data_path = os.path.join(self.data_dir, f\'VUA_Verb/train.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n    def get_verb_val(self):\n        data_path = os.path.join(self.data_dir, f\'VUA_Verb/val.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n    def get_verb_test(self):\n        data_path = os.path.join(self.data_dir, f\'VUA_Verb/test.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n    def get_trofi(self):\n        data_path = os.path.join(self.data_dir, f\'TroFi/TroFi.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n    def get_mohx(self):\n        data_path = os.path.join(self.data_dir, f\'MOH-X/MOH-X.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n    def get_acad(self):\n        data_path = os.path.join(self.data_dir, f\'VUA_All/genre/acad.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n    def get_conv(self):\n        data_path = os.path.join(self.data_dir, f\'VUA_All/genre/conv.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n    def get_fict(self):\n        data_path = os.path.join(self.data_dir, f\'VUA_All/genre/fict.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n    def get_news(self):\n        data_path = os.path.join(self.data_dir, f\'VUA_All/genre/news.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n    def get_adj(self):\n        data_path = os.path.join(self.data_dir, f\'VUA_All/pos/adj.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n    def get_adv(self):\n        data_path = os.path.join(self.data_dir, f\'VUA_All/pos/adv.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n    def get_noun(self):\n        data_path = os.path.join(self.data_dir, f\'VUA_All/pos/noun.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n    def get_verb(self):\n        data_path = os.path.join(self.data_dir, f\'VUA_All/pos/verb.tsv\')\n        data = self._get_pair_examples(data_path)\n        return data\n\n\nif __name__ == \'__main__\':\n    from configs.default import get_config\n    import argparse\n\n    def parse_option():\n        parser = argparse.ArgumentParser(description=\'Train on VUA ALL\')\n        parser.add_argument(\'--cfg\', type=str, default=\'./configs/vua_all.yaml\', metavar="FILE",\n                            help=\'path to config file\')\n        parser.add_argument(\'--gpu\', default=\'0\', type=str, help=\'gpu device ids\')\n        parser.add_argument(\'--seed\', default=\'4\', type=int, help=\'random seed\')\n        parser.add_argument(\'--eval\', action=\'store_true\', help="evaluation only")\n        parser.add_argument(\'--wsd_threshold\', default=3, type=int)\n        parser.add_argument(\'--log\', default=\'log_trofi\', type=str)\n        args, unparsed = parser.parse_known_args()\n        config = get_config(args)\n\n        return config\n\n    args = parse_option()\n    processor = WSD_Processor(args)\n    test_data = processor.get_wsd_toy()\n\n    mp = MD_Processor(args)\n    mohx_data = mp.get_mohx()\n    for item in mohx_data:\n        print(item)\n\n', 'data_utils.py': 'from torch.utils.data import Dataset, Sampler, RandomSampler\nimport torch\nimport math\n\n\nclass dataset(Dataset):\n    """wrap in PyTorch Dataset"""\n\n    def __init__(self, examples):\n        super(dataset, self).__init__()\n        self.examples = examples\n\n    def __getitem__(self, idx):\n        return self.examples[idx]\n\n    def __len__(self):\n        return len(self.examples)\n\n\nclass BalanceSampler(Sampler):\n    """\n    a self-defined dataset sampler. It can make sure that two dataset are sampled by their size ratio.\n\n    dataset: (pytorch concat dataset) a concat dataset of 2 separate datasets\n    batch_size: (int) the batch size for the concat dataset\n\n    """\n\n    def __init__(self, dataset, batch_size):\n        super(BalanceSampler, self).__init__(dataset)\n        self.dataset = dataset\n        self.batch_size = batch_size\n        self.dataset0_size = len(dataset.datasets[0])\n        self.dataset1_size = len(dataset.datasets[1])\n        if self.dataset0_size != self.dataset1_size:\n            raise ValueError("two datasets must have equal samples.")\n        self.number_of_datasets = len(dataset.datasets)\n\n    def __len__(self):\n        return self.dataset0_size + self.dataset1_size\n\n    def __iter__(self):\n        samplers_list = []\n        sampler_iterators = []\n        for dataset_idx in range(self.number_of_datasets):\n            cur_dataset = self.dataset.datasets[dataset_idx]\n            sampler = RandomSampler(cur_dataset)\n            samplers_list.append(sampler)\n            cur_sampler_iterator = sampler.__iter__()\n            sampler_iterators.append(cur_sampler_iterator)\n            # for item in sampler:\n            #     print(item)\n\n        sample_cnt_0 = self.batch_size // 2\n        sample_cnt_1 = self.batch_size // 2\n        sample_cnts = [sample_cnt_0, sample_cnt_1]\n        batch_cnt = math.ceil((self.dataset0_size + self.dataset1_size) / self.batch_size)  # 一共需要迭代多少个batch\n\n        push_index_val = [0] + self.dataset.cumulative_sizes[:-1]\n        sample_batch_indices = []  # 最后存储的索引\n\n        for batch_idx in range(batch_cnt):\n            cur_samples = []\n            if batch_idx != batch_cnt - 1:  # 每一个batch按比例采样batch_size个样本\n                for dataset_idx, sample_cnt in zip(range(self.number_of_datasets), sample_cnts):\n                    for i in range(sample_cnt):\n                        try:\n                            cur_sample = sampler_iterators[dataset_idx].__next__()\n                            cur_sample = cur_sample + push_index_val[dataset_idx]  # 原来的下标和合并后数据集的下标不一致\n                            cur_samples.append(cur_sample)\n                        except StopIteration:\n                            sampler_iterator = samplers_list[dataset_idx].__iter__()\n                            cur_sample_org = sampler_iterator.__next__()\n                            cur_sample = cur_sample_org + push_index_val[dataset_idx]\n                            cur_samples.append(cur_sample)\n            else:  # 最后一个batch直接迭代到迭代器的末尾\n                for dataset_idx in range(self.number_of_datasets):\n                    while True:\n                        try:\n                            cur_sample = sampler_iterators[dataset_idx].__next__()\n                            cur_sample = cur_sample + push_index_val[dataset_idx]  # 原来的下标和合并后数据集的下标不一致\n                            cur_samples.append(cur_sample)\n                        except StopIteration:\n                            break\n            sample_batch_indices.extend(cur_samples)\n        return iter(sample_batch_indices)\n\n\ndef collate_fn(examples):\n    input_ids, type_ids, att_mask, labels, domains = map(list, zip(*examples))\n\n    input_ids = torch.tensor(input_ids, dtype=torch.long)\n    type_ids = torch.tensor(type_ids, dtype=torch.long)\n    att_mask = torch.tensor(att_mask, dtype=torch.long)\n    labels = torch.tensor(labels, dtype=torch.long)\n    domains = torch.tensor(domains, dtype=torch.long)\n\n    return input_ids, type_ids, att_mask, labels, domains\n', 'train_val.py': 'import time\nfrom tqdm import tqdm\nimport torch\nimport torch.nn as nn\nfrom utils import overall_performance\nimport numpy as np\nimport random\n\n\ndef train(epoch, model, loss_fn_md, loss_fn_wsd, loss_fn_task, optimizer, train_loader, total_steps, LamdaSetter, args, scheduler=None):\n    epoch_start_time = time.time()\n    model.train()\n    tr_wsd_loss = 0  # training loss in current epoch\n    tr_md_loss = 0\n    tr_global_loss = 0\n    tr_local_loss = 0\n    # ! training\n    for step, batch in enumerate(tqdm(train_loader, desc=\'Iteration\')):\n        # unpack batch data\n        batch = tuple(t.cuda() for t in batch)\n        input_ids, type_ids, att_mask, labels, domains = batch  # domains: [1 x n]\n        adv_lmbd = LamdaSetter.set_lambda_(step + epoch * len(train_loader), total_steps)\n        half_batch_size = int(0.5 * len(labels))\n\n        # compute logits\n        wsd_out, md_out, global_out, wsd_local_out, md_local_out = model(input_ids, type_ids, att_mask, adv_lmbd)\n\n        wsd_labels = labels[:half_batch_size]\n        md_labels = labels[half_batch_size:]\n        wsd_domains = domains[:half_batch_size]\n        md_domains = domains[half_batch_size:]\n\n        # compute loss\n        wsd_loss = loss_fn_wsd(wsd_out, wsd_labels)\n        wsd_local_loss0 = loss_fn_task(wsd_local_out[0], wsd_domains)\n        wsd_local_loss1 = loss_fn_task(wsd_local_out[1], wsd_domains)\n\n        md_loss = loss_fn_md(md_out, md_labels)\n        global_loss = loss_fn_task(global_out, domains)\n\n        md_local_loss0 = loss_fn_task(md_local_out[0], md_domains)\n        md_local_loss1 = loss_fn_task(md_local_out[1], md_domains)\n\n        local_loss = 0.3 * wsd_local_loss0 + 0.3 * wsd_local_loss1 + md_local_loss0 + md_local_loss1\n\n        tr_wsd_loss += wsd_loss.item()\n        tr_md_loss += md_loss.item()\n        tr_global_loss += global_loss.item()\n        tr_local_loss += local_loss.item()\n\n        loss = md_loss + args.TRAIN.wsd_weight * wsd_loss + args.TRAIN.global_weight * global_loss + args.TRAIN.local_weight * local_loss\n\n        # back propagation\n        loss.backward()\n        # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n        optimizer.step()\n        if scheduler is not None:\n            scheduler.step()  # adjusting learning rate\n        optimizer.zero_grad()\n        step += 1\n\n    timing = time.time() - epoch_start_time\n    cur_lr = optimizer.param_groups[0]["lr"]\n    print(f"Timing: {timing}, Epoch: {epoch + 1}\\n"\n          f"md loss: {tr_md_loss}, "\n          f"wsd loss: {tr_wsd_loss}\\n"\n          f"global loss: {tr_global_loss}, "\n          f"local loss: {tr_local_loss}, "\n          f"current learning rate {cur_lr}")\n\n\ndef val(model, val_loader, domain_idx):\n    # make sure to open the eval mode.\n    model.eval()\n\n    # prepare loss function\n    loss_fn = nn.CrossEntropyLoss()\n\n    val_loss = 0\n    val_preds = []\n    val_labels = []\n    for batch in val_loader:\n        # unpack batch data\n        batch = tuple(t.cuda() for t in batch)\n        input_ids, type_ids, att_mask, labels, domains = batch\n\n        with torch.no_grad():\n            # compute logits\n            out = model(input_ids, type_ids, att_mask, domain_idx=domain_idx)\n            # get the prediction labels\n            preds = torch.max(out.data, 1)[1].cpu().numpy().tolist()  # prediction labels [1, batch_size]\n            # compute loss\n            loss = loss_fn(out, labels)\n            val_loss += loss.item()\n\n            labels = labels.cpu().numpy().tolist()  # ground truth labels [1, batch_size]\n            val_labels.extend(labels)\n            val_preds.extend(preds)\n    print(f"val loss: {val_loss}")\n\n    # get overall performance\n    val_acc, val_prec, val_recall, val_f1 = overall_performance(val_labels, val_preds)\n    return val_acc, val_prec, val_recall, val_f1\n\n\ndef set_random_seeds(seed):\n    """\n    set random seed\n    """\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.backends.cudnn.deterministic = True\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed(seed)\n', 'utils.py': 'import os\nimport errno\nimport os.path as osp\nimport json\nimport sys\nimport numpy as np\nfrom sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score\n\n\ndef mkdir_if_missing(directory):\n    if not osp.exists(directory):\n        try:\n            os.makedirs(directory)\n        except OSError as e:\n            if e.errno != errno.EEXIST:\n                raise\n\n\ndef read_json(fpath):\n    with open(fpath, \'r\') as f:\n        obj = json.load(f)\n    return obj\n\n\ndef write_json(obj, fpath):\n    mkdir_if_missing(osp.dirname(fpath))\n    with open(fpath, \'w\') as f:\n        json.dump(obj, f, indent=4, separators=(\',\', \': \'))\n\n\nclass Logger(object):\n    """\n    Write console output to external text file.\n    Code imported from https://github.com/Cysu/open-reid/blob/master/reid/utils/logging.py.\n    """\n    def __init__(self, fpath=None):\n        self.console = sys.stdout\n        self.file = None\n        if fpath is not None:\n            mkdir_if_missing(os.path.dirname(fpath))\n            self.file = open(fpath, \'w\')\n\n    def __del__(self):\n        self.close()\n\n    def __enter__(self):\n        pass\n\n    def __exit__(self, *args):\n        self.close()\n\n    def write(self, msg):\n        self.console.write(msg)\n        if self.file is not None:\n            self.file.write(msg)\n\n    def flush(self):\n        self.console.flush()\n        if self.file is not None:\n            self.file.flush()\n            os.fsync(self.file.fileno())\n\n    def close(self):\n        self.console.close()\n        if self.file is not None:\n            self.file.close()\n\n\ndef overall_performance(labels, preds):\n    """\n\n    :param labels: ground truth labels\n    :param preds: prediction labels\n    :return: accuracy, precision, recall, f1\n    """\n    accuracy = accuracy_score(labels, preds)\n    precision = precision_score(labels, preds)\n    recall = recall_score(labels, preds)\n    f1 = f1_score(labels, preds)\n\n    print("{:15}{:<.3f}".format(\'accuracy:\', accuracy))\n    print("{:15}{:<.3f}".format(\'precision:\', precision))\n    print("{:15}{:<.3f}".format(\'recall:\', recall))\n    print("{:15}{:<.3f}".format(\'f1\', f1))\n\n    return accuracy, precision, recall, f1\n\n\nclass Lamda():\n    def __init__(self, lo, hi):\n        self.lo = lo\n        self.hi = hi\n        self.m = 2 * (hi - lo)\n        self.n = hi - 2 * lo\n\n    def set_lambda_(self, cur_step, total_step):\n        p = cur_step / total_step\n        return self.m / (1 + np.exp(-10 * p)) - self.n', 'main_vua.py': 'import os\nimport os.path as osp\nimport sys\nimport argparse\nfrom torch.utils.data import DataLoader, ConcatDataset\nfrom transformers import AdamW, get_linear_schedule_with_warmup\nimport torch.nn as nn\nimport torch\nimport time\nimport random\n\nfrom model import DeBERTa_base\nfrom data_utils import dataset, collate_fn, BalanceSampler\nfrom data_process import WSD_Processor, MD_Processor\nfrom utils import Logger, Lamda\nfrom configs.default import get_config\nfrom train_val import train, val, set_random_seeds\n\n\ndef parse_option():\n    parser = argparse.ArgumentParser(description=\'Train on VUA dataset\')\n    parser.add_argument(\'--cfg\', type=str, default=\'./configs/vua_verb.yaml\', metavar="FILE",\n                        help=\'path to config file\')\n    parser.add_argument(\'--gpu\', default=\'0\', type=str, help=\'gpu device ids\')\n    parser.add_argument(\'--seed\', default=4, type=int, help=\'random seed\')\n    parser.add_argument(\'--eval\', action=\'store_true\', help="evaluation only")\n    parser.add_argument(\'--task\', default=\'verb\', type=str, help=\'train on vua verb or vua all\')\n    args, unparsed = parser.parse_known_args()\n    config = get_config(args)\n\n    return config\n\n\ndef main(args):\n    os.environ[\'CUDA_VISIBLE_DEVICES\'] = args.gpu\n    sys.stdout = Logger(osp.join(args.TRAIN.output,\n                                 f\'{args.task}_{time.strftime("%Y-%m-%d_%H:%M:%S", time.localtime())}.txt\'))\n    print(args)\n    set_random_seeds(args.seed)\n\n    wp = WSD_Processor(args)\n    mp = MD_Processor(args)\n\n    if args.eval_mode:\n        print("Evaluate only")\n        if args.task == \'all\':\n            model = DeBERTa_base(args)\n            model.cuda()\n            model.load_state_dict(torch.load(\'./best_check.pth\'))\n            test_data = mp.get_all_test()\n            acad_data = mp.get_acad()\n            conv_data = mp.get_conv()\n            fict_data = mp.get_fict()\n            news_data = mp.get_news()\n            adj_data = mp.get_adj()\n            adv_data = mp.get_adv()\n            noun_data = mp.get_noun()\n            verb_data = mp.get_verb()\n\n            test_set = dataset(test_data)\n            acad_set = dataset(acad_data)\n            conv_set = dataset(conv_data)\n            fict_set = dataset(fict_data)\n            news_set = dataset(news_data)\n            adj_set = dataset(adj_data)\n            adv_set = dataset(adv_data)\n            noun_set = dataset(noun_data)\n            verb_set = dataset(verb_data)\n\n            test_loader = DataLoader(test_set, batch_size=args.TRAIN.val_batch_size, shuffle=False, collate_fn=collate_fn)\n            acad_loader = DataLoader(acad_set, batch_size=args.TRAIN.val_batch_size, shuffle=False, collate_fn=collate_fn)\n            conv_loader = DataLoader(conv_set, batch_size=args.TRAIN.val_batch_size, shuffle=False, collate_fn=collate_fn)\n            fict_loader = DataLoader(fict_set, batch_size=args.TRAIN.val_batch_size, shuffle=False, collate_fn=collate_fn)\n            news_loader = DataLoader(news_set, batch_size=args.TRAIN.val_batch_size, shuffle=False, collate_fn=collate_fn)\n            adj_loader = DataLoader(adj_set, batch_size=args.TRAIN.val_batch_size, shuffle=False, collate_fn=collate_fn)\n            adv_loader = DataLoader(adv_set, batch_size=args.TRAIN.val_batch_size, shuffle=False, collate_fn=collate_fn)\n            noun_loader = DataLoader(noun_set, batch_size=args.TRAIN.val_batch_size, shuffle=False, collate_fn=collate_fn)\n            verb_loader = DataLoader(verb_set, batch_size=args.TRAIN.val_batch_size, shuffle=False, collate_fn=collate_fn)\n\n            # transfer learning part\n            trofi_data = mp.get_trofi()\n            trofi_set = dataset(trofi_data)\n            trofi_loader = DataLoader(trofi_set, batch_size=args.TRAIN.val_batch_size, shuffle=False, collate_fn=collate_fn)\n            mohx_data = mp.get_mohx()\n            mohx_set = dataset(mohx_data)\n            mohx_loader = DataLoader(mohx_set, batch_size=args.TRAIN.val_batch_size, shuffle=False, collate_fn=collate_fn)\n\n            print(\'-------test-------\')\n            val(model, test_loader, domain_idx=1)\n            print(\'-------acad-------\')\n            val(model, acad_loader, domain_idx=1)\n            print(\'-------conv-------\')\n            val(model, conv_loader, domain_idx=1)\n            print(\'-------fict-------\')\n            val(model, fict_loader, domain_idx=1)\n            print(\'-------news-------\')\n            val(model, news_loader, domain_idx=1)\n            print(\'-------adj-------\')\n            val(model, adj_loader, domain_idx=1)\n            print(\'-------adv-------\')\n            val(model, adv_loader, domain_idx=1)\n            print(\'-------noun-------\')\n            val(model, noun_loader, domain_idx=1)\n            print(\'-------verb-------\')\n            val(model, verb_loader, domain_idx=1)\n            print(\'-------trofi-------\')\n            val(model, trofi_loader, domain_idx=1)\n            print(\'-------mohx-------\')\n            val(model, mohx_loader, domain_idx=1)\n            return\n        if args.task == \'verb\':\n            model = DeBERTa_base(args)\n            model.cuda()\n            model.load_state_dict(torch.load(\'./best_check.pth\'))\n            test_data = mp.get_verb_test()\n            test_set = dataset(test_data)\n            test_loader = DataLoader(test_set, batch_size=args.TRAIN.val_batch_size, shuffle=False, collate_fn=collate_fn)\n            print(\'-------test-------\')\n            val(model, test_loader, domain_idx=1)\n\n            return\n\n    if args.task == \'verb\':\n        md_train_data = mp.get_verb_train()\n        md_test_data = mp.get_verb_test()\n        md_val_data = mp.get_verb_val()\n    else:\n        md_train_data = mp.get_all_train()\n        md_val_data = mp.get_all_val()\n        md_test_data = mp.get_all_test()\n\n    md_train_set = dataset(md_train_data)\n    md_val_set = dataset(md_val_data)\n    md_test_set = dataset(md_test_data)\n\n    wsd_train_data = wp.get_wsd_train()\n    if len(wsd_train_data) >= len(md_train_data):\n        wsd_train_data = wsd_train_data[:len(md_train_data)]\n    random.shuffle(wsd_train_data)\n    wsd_test_data = wp.get_wsd_test()\n\n    wsd_train_set = dataset(wsd_train_data)\n    wsd_test_set = dataset(wsd_test_data)\n\n    # concat wsd dataset and metaphor detection dataset\n    # make sure to put the smaller dataset as the first element\n    train_set = ConcatDataset([wsd_train_set, md_train_set])\n\n    train_loader = DataLoader(dataset=train_set,\n                              sampler=BalanceSampler(dataset=train_set, batch_size=args.TRAIN.train_batch_size),\n                              batch_size=args.TRAIN.train_batch_size,\n                              collate_fn=collate_fn,\n                              shuffle=False)\n\n    wsd_test_loader = DataLoader(dataset=wsd_test_set,\n                                 batch_size=args.TRAIN.val_batch_size,\n                                 collate_fn=collate_fn,\n                                 shuffle=False)\n    md_val_loader = DataLoader(dataset=md_val_set,\n                               batch_size=args.TRAIN.val_batch_size,\n                               collate_fn=collate_fn,\n                               shuffle=False)\n    md_test_loader = DataLoader(dataset=md_test_set,\n                                batch_size=args.TRAIN.val_batch_size,\n                                collate_fn=collate_fn,\n                                shuffle=False)\n\n    # load model\n    model = DeBERTa_base(args)\n    model.cuda()\n\n    # prepare optimizer\n    optimizer = AdamW(model.parameters(), lr=args.TRAIN.lr, weight_decay=args.TRAIN.weight_decay)\n\n    # prepare loss function\n    loss_fn_wsd = nn.CrossEntropyLoss()\n    loss_fn_md = nn.CrossEntropyLoss(weight=torch.Tensor([1, args.TRAIN.md_class_weight]).cuda())\n    loss_fn_task = nn.CrossEntropyLoss(weight=torch.Tensor([1, args.TRAIN.wsd_class_weight]).cuda())\n\n    LamdaSetter = Lamda(lo=args.TRAIN.lambda_lo, hi=args.TRAIN.lambda_hi)\n    total_steps = len(train_loader) * args.TRAIN.train_epochs\n    if args.task == \'all\':\n        scheduler = get_linear_schedule_with_warmup(\n            optimizer,\n            num_warmup_steps=int(args.TRAIN.warmup_epochs * len(train_loader)),\n            num_training_steps=total_steps,\n        )\n    else:\n        scheduler = None\n    best_f1 = 0\n    for epoch in range(args.TRAIN.train_epochs):\n        print(\'===== Start training: epoch {} =====\'.format(epoch + 1))\n        train(epoch, model, loss_fn_md, loss_fn_wsd, loss_fn_task, optimizer, train_loader, total_steps, LamdaSetter, args, scheduler=scheduler)\n        a, p, r, f1 = val(model, md_val_loader, domain_idx=1)\n        val(model, md_test_loader, domain_idx=1)\n        val(model, wsd_test_loader, domain_idx=0)\n        if f1 > best_f1:\n            best_f1 = f1\n            torch.save(model.state_dict(), \'./best_check.pth\')\n\n\nif __name__ == \'__main__\':\n    args = parse_option()\n    main(args)\n', 'main_kf.py': 'import os\nimport os.path as osp\nimport random\nimport sys\nimport argparse\nimport torch\nfrom torch.utils.data import DataLoader, ConcatDataset\nfrom transformers import AdamW\nimport torch.nn as nn\nimport time\n\nfrom model import DeBERTa_base\nfrom data_utils import dataset, collate_fn, BalanceSampler\nfrom data_process import WSD_Processor, MD_Processor\nfrom utils import Logger, Lamda\nfrom configs.default import get_config\nfrom train_val import train, val, set_random_seeds\n\n\ndef parse_option():\n    parser = argparse.ArgumentParser(description=\'Train on MOH-X dataset, do cross validation\')\n    parser.add_argument(\'--cfg\', type=str, default=\'./configs/mohx.yaml\', metavar="FILE",\n                        help=\'path to config file\')\n    parser.add_argument(\'--gpu\', default=\'0\', type=str, help=\'gpu device ids\')\n    parser.add_argument(\'--seed\', default=4, type=int, help=\'random seed\')\n    parser.add_argument(\'--eval\', action=\'store_true\', help="evaluation only")\n    parser.add_argument(\'--task\', default=\'mohx\', type=str, help=\'train on vua verb or vua all\')\n    args, unparsed = parser.parse_known_args()\n    config = get_config(args)\n\n    return config\n\n\ndef get_kfold_data(k, i, raw_data):\n    fold_size = len(raw_data) // k\n\n    val_start = i * fold_size\n    if i != k - 1:\n        val_end = (i + 1) * fold_size\n        val_raw = raw_data[val_start:val_end]\n        tr_raw = raw_data[0:val_start] + raw_data[val_end:]\n    else:\n        val_raw = raw_data[val_start:]\n        tr_raw = raw_data[0:val_start]\n\n    return tr_raw, val_raw\n\n\ndef main(args):\n    os.environ[\'CUDA_VISIBLE_DEVICES\'] = args.gpu\n    sys.stdout = Logger(osp.join(args.TRAIN.output,\n                                 f\'{args.task}_{time.strftime("%Y-%m-%d_%H:%M:%S", time.localtime())}.txt\'))\n    print(args)\n    set_random_seeds(args.seed)\n\n    wp = WSD_Processor(args)\n    mp = MD_Processor(args)\n\n    if args.task == \'mohx\':\n        md_data = mp.get_mohx()\n    else:\n        md_data = mp.get_trofi()\n    random.shuffle(md_data)\n    wsd_train_data = wp.get_wsd_train()\n    random.shuffle(wsd_train_data)\n    wsd_test_data = wp.get_wsd_test()\n\n    accs = 0\n    pres = 0\n    recs = 0\n    f1s = 0\n    for i in range(10):\n        print(\'*\' * 20, f"training on fold #{i + 1}", \'*\' * 20)\n        md_train_data, md_test_data = get_kfold_data(10, i, md_data)\n        md_train_set = dataset(md_train_data)\n        md_test_set = dataset(md_test_data)\n\n        if len(wsd_train_data) > len(md_train_data):\n            wsd_train_data = wsd_train_data[:len(md_train_data)]\n        wsd_train_set = dataset(wsd_train_data)\n        wsd_test_set = dataset(wsd_test_data)\n\n        train_set = ConcatDataset([wsd_train_set, md_train_set])\n\n        train_loader = DataLoader(dataset=train_set,\n                                  sampler=BalanceSampler(dataset=train_set, batch_size=args.TRAIN.train_batch_size),\n                                  batch_size=args.TRAIN.train_batch_size,\n                                  collate_fn=collate_fn,\n                                  shuffle=False)\n\n        wsd_test_loader = DataLoader(dataset=wsd_test_set,\n                                     batch_size=args.TRAIN.val_batch_size,\n                                     collate_fn=collate_fn,\n                                     shuffle=False)\n\n        md_test_loader = DataLoader(dataset=md_test_set,\n                                    batch_size=args.TRAIN.val_batch_size,\n                                    collate_fn=collate_fn,\n                                    shuffle=False)\n        # load model\n        model = DeBERTa_base(args)\n        model.cuda()\n\n        # prepare optimizer\n        optimizer = AdamW(model.parameters(), lr=args.TRAIN.lr, weight_decay=args.TRAIN.weight_decay)\n\n        # prepare loss function\n        loss_fn_wsd = nn.CrossEntropyLoss()\n        loss_fn_md = nn.CrossEntropyLoss(weight=torch.Tensor([1, args.TRAIN.md_class_weight]).cuda())\n        loss_fn_task = nn.CrossEntropyLoss(weight=torch.Tensor([1, args.TRAIN.wsd_class_weight]).cuda())\n\n        LamdaSetter = Lamda(lo=args.TRAIN.lambda_lo, hi=args.TRAIN.lambda_hi)\n        total_steps = len(train_loader) * args.TRAIN.train_epochs\n\n        best_acc = -1\n        best_pre = -1\n        best_rec = -1\n        best_f1 = -1\n        for epoch in range(args.TRAIN.train_epochs):\n            print(\'===== Start training: epoch {} =====\'.format(epoch + 1))\n            train(epoch, model, loss_fn_md, loss_fn_wsd, loss_fn_task, optimizer, train_loader, total_steps, LamdaSetter, args)\n            t_a, t_p, t_r, t_f1 = val(model, md_test_loader, domain_idx=1)\n            val(model, wsd_test_loader, domain_idx=0)\n            if t_f1 > best_f1:\n                best_acc, best_pre, best_rec, best_f1 = t_a, t_p, t_r, t_f1\n        accs += best_acc\n        pres += best_pre\n        recs += best_rec\n        f1s += best_f1\n\n    print(\'average result:\')\n    print(accs / 10)\n    print(pres / 10)\n    print(recs / 10)\n    print(f1s / 10)\n\n\nif __name__ == \'__main__\':\n    args = parse_option()\n    main(args)\n', 'configs/default.py': 'from yacs.config import CfgNode as CN\n\n_C = CN()\n\n"""\n    data settings\n"""\n_C.DATA = CN()\n_C.DATA.wsd_threshold = 3\n_C.DATA.data_dir = \'./data\'\n_C.DATA.max_len = 150\n_C.DATA.plm = \'./deberta-v3-base\'\n"""\n    model settings\n"""\n_C.MODEL = CN()\n_C.MODEL.num_classes = 2\n_C.MODEL.num_tasks = 2\n# embedding dim\n_C.MODEL.embed_dim = 768\n# drop out rate\n_C.MODEL.dropout = 0.2\n\n\'\'\'\n    training settings\n\'\'\'\n_C.TRAIN = CN()\n_C.TRAIN.train_batch_size = 32\n_C.TRAIN.val_batch_size = 32\n_C.TRAIN.lr = 3e-5\n_C.TRAIN.train_epochs = 5\n_C.TRAIN.warmup_epochs = 2\n# the directory to save the training logs\n_C.TRAIN.output = \'./data/logs\'\n_C.TRAIN.md_class_weight = 4\n_C.TRAIN.wsd_class_weight = 4\n_C.TRAIN.weight_decay = 0.01\n_C.TRAIN.wsd_weight = 0.2\n_C.TRAIN.local_weight = 0.1\n_C.TRAIN.global_weight = 0.1\n_C.TRAIN.lambda_lo = 0.1\n_C.TRAIN.lambda_hi = 0.8\n_C.gpu = \'0\'\n_C.seed = 4\n# do eval only\n_C.eval_mode = False\n_C.log = \'log_test\'\n_C.task = \'verb\'\n\n\ndef update_config(config, args):\n    config.defrost()\n\n    print(\'=> merge config from {}\'.format(args.cfg))\n    config.merge_from_file(args.cfg)\n\n    if args.gpu:\n        config.gpu = args.gpu\n\n    if args.seed:\n        config.seed = args.seed\n\n    if args.eval:\n        config.eval_mode = True\n\n    if args.task:\n        config.task = args.task\n    config.freeze()\n\n\ndef get_config(args):\n    """Get a yacs CfgNode object with default values."""\n    config = _C.clone()\n    update_config(config, args)\n\n    return config\n', 'configs/vua_all.yaml': 'TRAIN:\n  md_class_weight: 4\n  wsd_class_weight: 1\n  train_epochs: 8\n  train_batch_size: 64\n  val_batch_size: 64\n  lambda_lo: 0.1\n  lambda_hi: 0.8\nDATA:\n  wsd_threshold: 3\nMODEL:\n  dropout: 0.0\n', 'configs/vua_verb.yaml': 'TRAIN:\n  train_batch_size: 64\n  val_batch_size: 64\n  md_class_weight: 1\n  wsd_class_weight: 1\nMODEL:\n  dropout: 0.0\nDATA:\n  wsd_threshold: 8', 'configs/mohx.yaml': 'TRAIN:\n  train_batch_size: 32\n  val_batch_size: 32\n  md_class_weight: 1\n  wsd_class_weight: 1\n  lr: 2e-5\nMODEL:\n  dropout: 0.0\nDATA:\n  wsd_threshold: 8', 'configs/trofi.yaml': 'TRAIN:\n  train_batch_size: 64\n  val_batch_size: 64\n  md_class_weight: 1\n  wsd_class_weight: 1\n  lr: 1e-5\nMODEL:\n  dropout: 0.0\nDATA:\n  wsd_threshold: 8', 'runner.py': '"""VUA All, author model/processors/sampler/train/val, full epoch continuation."""\nimport os\nos.environ[\'USE_TF\'] = \'0\'\nos.environ[\'USE_FLAX\'] = \'0\'\nos.environ[\'CUBLAS_WORKSPACE_CONFIG\'] = \':4096:8\'\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport platform\nimport random\nimport shutil\nimport sys\nimport time\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nimport numpy as np\nimport torch\nimport torch.utils.checkpoint\nimport transformers\nimport tokenizers\nimport sklearn\nfrom transformers import AdamW, get_linear_schedule_with_warmup\nfrom torch.utils.data import DataLoader, ConcatDataset\n\nfrom checkpointing import (atomic_json, atomic_save, rng_state, restore_rng,\n                           latest_checkpoint, load_checkpoint, save_epoch)\nfrom configs.default import get_config\nfrom model import DeBERTa_base\nfrom data_process import WSD_Processor, MD_Processor\nfrom data_utils import dataset, collate_fn, BalanceSampler\nfrom train_val import train, val, set_random_seeds\nfrom utils import Lamda\n\n\nclass SessionDeadline(Exception):\n    pass\n\n\nclass TimedLoader:\n    """Keep author batch order; return normally from the notebook before timeout."""\n    def __init__(self, loader, deadline):\n        self.loader, self.deadline = loader, deadline\n\n    def __len__(self):\n        return len(self.loader)\n\n    def __iter__(self):\n        for batch in self.loader:\n            if time.monotonic() >= self.deadline:\n                raise SessionDeadline()\n            yield batch\n\n\ndef digest(path):\n    value = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n            value.update(block)\n    return value.hexdigest()\n\n\ndef tree_hashes(root, pattern):\n    return {str(p.relative_to(root)).replace(\'\\\\\', \'/\'): digest(p)\n            for p in sorted(Path(root).rglob(pattern)) if p.is_file()}\n\n\ndef runtime():\n    return dict(python=platform.python_version(), torch=torch.__version__,\n                transformers=transformers.__version__, numpy=np.__version__,\n                tokenizers=tokenizers.__version__, sklearn=sklearn.__version__,\n                cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),\n                gpu=torch.cuda.get_device_name(0), devices=torch.cuda.device_count(),\n                cudnn_deterministic=torch.backends.cudnn.deterministic,\n                cudnn_benchmark=torch.backends.cudnn.benchmark,\n                matmul_tf32=torch.backends.cuda.matmul.allow_tf32,\n                cudnn_tf32=torch.backends.cudnn.allow_tf32,\n                deterministic_algorithms=torch.are_deterministic_algorithms_enabled())\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--work\', type=Path, required=True)\n    parser.add_argument(\'--seed\', type=int, default=4)\n    parser.add_argument(\'--max-session-epochs\', type=int, default=3)\n    parser.add_argument(\'--seconds-left\', type=float, default=10.5 * 3600)\n    opts = parser.parse_args()\n    if opts.max_session_epochs < 1 or opts.seconds_left <= 0:\n        parser.error(\'No session time left or invalid epoch limit\')\n    work = opts.work.resolve()\n    os.chdir(work)\n    run = work / \'run\'\n    run.mkdir(exist_ok=True)\n    checkpoint = latest_checkpoint(run)\n    deadline = time.monotonic() + opts.seconds_left\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Enable a Kaggle GPU; TPU is not supported by this CUDA code\')\n    if transformers.__version__ != \'4.18.0\' or np.__version__ != \'1.21.3\':\n        raise RuntimeError(\'Use the isolated Python environment installed by the notebook\')\n    args = get_config(SimpleNamespace(cfg=str(work / \'src/configs/vua_all.yaml\'),\n                                      gpu=\'0\', seed=opts.seed, eval=False, task=\'all\'))\n    args.defrost()\n    args.DATA.data_dir = str(work / \'data/data\')\n    args.DATA.plm = str(work / \'deberta-v3-base\')\n    args.TRAIN.output = str(run)\n    args.freeze()\n    set_random_seeds(args.seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.use_deterministic_algorithms(True, warn_only=True)\n    (run / \'config.yaml\').write_text(args.dump(), encoding=\'utf-8\')\n\n    def emit(event, **values):\n        value = dict(event=event, **values)\n        line = json.dumps(value, ensure_ascii=False, allow_nan=False)\n        print(line, flush=True)\n        with (run / \'events.jsonl\').open(\'a\', encoding=\'utf-8\') as stream:\n            stream.write(line + \'\\n\')\n\n    data_root = Path(args.DATA.data_dir)\n    data_hashes = tree_hashes(data_root, \'*.tsv\')\n    expected = {\'VUA_All/train.tsv\': 116622, \'VUA_All/val.tsv\': 38628,\n                \'VUA_All/test.tsv\': 50175, \'semcor/threshold/train_3.tsv\': 130808,\n                \'semcor/threshold/test_3.tsv\': 4081}\n    for name, count in expected.items():\n        with (data_root / name).open(encoding=\'utf-8-sig\', newline=\'\') as stream:\n            actual = sum(1 for _ in csv.DictReader(stream, delimiter=\'\\t\'))\n        if actual != count:\n            raise ValueError(\'%s: expected %s rows, got %s\' % (name, count, actual))\n    cache_path = run / \'features.pt\'\n    input_identity = dict(data=data_hashes, source=tree_hashes(work / \'src\', \'*.py\'),\n                          config=args.dump(), runtime=runtime(),\n                          backbone=tree_hashes(work / \'deberta-v3-base\', \'*\'))\n    if (run / \'identity.json\').exists():\n        previous_identity = json.loads((run / \'identity.json\').read_text(encoding=\'utf-8\'))\n        for key in input_identity:\n            if previous_identity[key] != input_identity[key]:\n                raise ValueError(\'Resume rejected: changed \' + key)\n        if not cache_path.exists() or previous_identity[\'features_sha256\'] != digest(cache_path):\n            raise ValueError(\'Missing or modified tokenized dataset snapshot\')\n    elif checkpoint is not None:\n        raise ValueError(\'Checkpoint present without identity.json\')\n\n    if cache_path.exists():\n        features = torch.load(cache_path, map_location=\'cpu\', weights_only=False)\n        if features[\'input_identity\'] != input_identity:\n            raise ValueError(\'Tokenized dataset cache belongs to a different run\')\n        # Also replay preprocessing RNG on a fresh run with cached features.\n        restore_rng(features[\'before_model_rng\'])\n    else:\n        # Preserve main_vua.py processing/truncation/shuffle order exactly.\n        wp, mp = WSD_Processor(args), MD_Processor(args)\n        md_train = mp.get_all_train()\n        md_dev = mp.get_all_val()\n        md_test = mp.get_all_test()\n        wsd_train = wp.get_wsd_train()\n        if len(wsd_train) >= len(md_train):\n            wsd_train = wsd_train[:len(md_train)]\n        random.shuffle(wsd_train)\n        wsd_test = wp.get_wsd_test()\n        features = dict(md_train=md_train, md_dev=md_dev, md_test=md_test,\n                        wsd_train=wsd_train, wsd_test=wsd_test,\n                        input_identity=input_identity, before_model_rng=rng_state())\n        atomic_save(cache_path, features)\n        mp.tokenizer.save_pretrained(run / \'tokenizer\')\n    train_set = ConcatDataset([dataset(features[\'wsd_train\']), dataset(features[\'md_train\'])])\n    train_loader = DataLoader(train_set, sampler=BalanceSampler(train_set, args.TRAIN.train_batch_size),\n                              batch_size=args.TRAIN.train_batch_size, collate_fn=collate_fn, shuffle=False)\n    eval_loaders = {name: DataLoader(dataset(features[key]), batch_size=args.TRAIN.val_batch_size,\n                                     collate_fn=collate_fn, shuffle=False)\n                    for name, key in [(\'bsd_test\', \'wsd_test\'), (\'md_dev\', \'md_dev\'), (\'md_test\', \'md_test\')]}\n    sizes = {name: len(features[name]) for name in (\'md_train\', \'md_dev\', \'md_test\', \'wsd_train\', \'wsd_test\')}\n    del features\n    total_steps = len(train_loader) * args.TRAIN.train_epochs\n    identity = dict(input_identity, features_sha256=digest(cache_path),\n                    steps_per_epoch=len(train_loader), total_steps=total_steps,\n                    gradient_checkpointing=True, mixed_precision=False)\n    atomic_json(run / \'identity.json\', identity)\n    model = DeBERTa_base(args).cuda()\n    # HF 4.18 uses PyTorch checkpoint() with preserve_rng_state=True by default.\n    # Only recompute activations; retain original FP32 batch and optimizer updates.\n    model.plm.gradient_checkpointing_enable()\n    if not model.plm.is_gradient_checkpointing:\n        raise RuntimeError(\'Backbone did not enable gradient checkpointing\')\n    optimizer = AdamW(model.parameters(), lr=args.TRAIN.lr, weight_decay=args.TRAIN.weight_decay)\n    scheduler = get_linear_schedule_with_warmup(\n        optimizer, num_warmup_steps=int(args.TRAIN.warmup_epochs * len(train_loader)),\n        num_training_steps=total_steps)\n    loss_wsd = torch.nn.CrossEntropyLoss()\n    loss_md = torch.nn.CrossEntropyLoss(weight=torch.tensor([1., args.TRAIN.md_class_weight]).cuda())\n    loss_task = torch.nn.CrossEntropyLoss(weight=torch.tensor([1., args.TRAIN.wsd_class_weight]).cuda())\n    lam = Lamda(args.TRAIN.lambda_lo, args.TRAIN.lambda_hi)\n    history, completed, best_epoch, best_f1 = [], 0, None, -1.0\n    if checkpoint is not None:\n        state = load_checkpoint(checkpoint, identity)\n        model.load_state_dict(state[\'model\'], strict=True)\n        optimizer.load_state_dict(state[\'optimizer\'])\n        scheduler.load_state_dict(state[\'scheduler\'])\n        history, completed = state[\'history\'], state[\'completed_epochs\']\n        best_epoch, best_f1 = state[\'best_epoch\'], state[\'best_f1\']\n        if scheduler.last_epoch != completed * len(train_loader):\n            raise ValueError(\'Scheduler progress changed\')\n        restore_rng(state[\'rng\'])\n        del state\n    emit(\'resume\' if checkpoint else \'fresh_start\', completed_epochs=completed,\n         next_epoch=completed + 1, optimizer_restored=checkpoint is not None,\n         best_epoch=best_epoch, best_dev_f1=best_f1, sizes=sizes,\n         steps_per_epoch=len(train_loader), lr=optimizer.param_groups[0][\'lr\'], environment=runtime())\n    session_epochs = 0\n    for epoch in range(completed, args.TRAIN.train_epochs):\n        estimated = max((record[\'seconds\'] for record in history), default=3 * 3600)\n        if (session_epochs >= opts.max_session_epochs or\n                time.monotonic() + estimated * 1.2 + 300 >= deadline):\n            emit(\'session_stop\', completed_epochs=completed, next_epoch=completed + 1,\n                 reason=\'Leave time for a complete epoch and checkpoint\')\n            break\n        # Every completed epoch is retained; reserve enough room for the next full save.\n        required = 3 * sum(t.numel() * t.element_size() for t in model.state_dict().values()) + 512 * 1024**2\n        if shutil.disk_usage(work).free < required:\n            raise RuntimeError(\'Not enough free disk for the next full epoch checkpoint\')\n        started = time.monotonic()\n        emit(\'epoch_start\', epoch=epoch + 1)\n        try:\n            train(epoch, model, loss_md, loss_wsd, loss_task, optimizer,\n                  TimedLoader(train_loader, deadline), total_steps, lam, args, scheduler=scheduler)\n            scores = {}\n            # Keep ALL original evaluations: DataLoader iterators consume torch RNG,\n            # so removing the two test passes changes future training batch order.\n            for name, domain in [(\'md_dev\', 1), (\'md_test\', 1), (\'bsd_test\', 0)]:\n                values = val(model, TimedLoader(eval_loaders[name], deadline), domain_idx=domain)\n                scores[name] = dict(zip((\'accuracy\', \'precision\', \'recall\', \'f1\'), map(float, values)))\n        except SessionDeadline:\n            # No partial state replaces a completed epoch. Exit normally so Output persists.\n            emit(\'session_stop\', completed_epochs=completed, next_epoch=completed + 1,\n                 reason=\'Time budget reached; replay unfinished epoch from last checkpoint\')\n            break\n        torch.cuda.synchronize()\n        record = dict(epoch=epoch + 1, seconds=time.monotonic() - started, **scores,\n                      lr=optimizer.param_groups[0][\'lr\'])\n        if scores[\'md_dev\'][\'f1\'] > best_f1:\n            best_f1, best_epoch = scores[\'md_dev\'][\'f1\'], epoch + 1\n        history.append(record)\n        path = save_epoch(run, epoch + 1, model, optimizer, scheduler, history, identity, best_epoch, best_f1)\n        completed, session_epochs = epoch + 1, session_epochs + 1\n        emit(\'checkpoint_saved\', epoch=completed, path=str(path), metrics=record)\n    result = dict(training_complete=completed == args.TRAIN.train_epochs,\n                  completed_epochs=completed, next_epoch=completed + 1 if completed < args.TRAIN.train_epochs else None,\n                  best_epoch=best_epoch, best_dev_f1=best_f1, session_epochs=session_epochs,\n                  best_checkpoint=(\'epochs/epoch_%02d.pt\' % best_epoch) if best_epoch else None,\n                  note=\'Best epoch selected by dev only; test passes retained to match author RNG order\')\n    if best_epoch is not None:\n        result[\'best_epoch_test\'] = history[best_epoch - 1][\'md_test\']\n    atomic_json(run / \'result.json\', result)\n    emit(\'complete\' if result[\'training_complete\'] else \'session_complete\', **result)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'checkpointing.py': '"""Epoch checkpoints. Only load these pickle files from your own trusted runs."""\nimport json\nimport os\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\n\ndef rng_state():\n    return dict(python=random.getstate(), numpy=np.random.get_state(),\n                torch=torch.get_rng_state(),\n                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])\n\n\ndef restore_rng(state):\n    random.setstate(state[\'python\'])\n    np.random.set_state(state[\'numpy\'])\n    torch.set_rng_state(state[\'torch\'])\n    if state[\'cuda\']:\n        if len(state[\'cuda\']) != torch.cuda.device_count():\n            raise ValueError(\'CUDA device count changed from checkpoint\')\n        torch.cuda.set_rng_state_all(state[\'cuda\'])\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    with temp.open(\'w\', encoding=\'utf-8\') as stream:\n        json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)\n        stream.write(\'\\n\')\n        stream.flush()\n        os.fsync(stream.fileno())\n    os.replace(temp, path)\n\n\ndef atomic_save(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    with temp.open(\'wb\') as stream:\n        torch.save(value, stream)\n        stream.flush()\n        os.fsync(stream.fileno())\n    os.replace(temp, path)\n\n\ndef latest_checkpoint(run_dir):\n    # A final .pt only appears after a complete, flushed save; ignore unfinished .tmp.\n    # Scan files so a crash after saving .pt but before latest.json also recovers.\n    paths = sorted((Path(run_dir) / \'epochs\').glob(\'epoch_*.pt\'))\n    return paths[-1] if paths else None\n\n\ndef load_checkpoint(path, identity):\n    state = torch.load(path, map_location=\'cpu\', weights_only=False)\n    if state.get(\'format\') != \'admul-epoch-v1\':\n        raise ValueError(\'Not a full checkpoint from this notebook\')\n    if state[\'identity\'] != identity:\n        changed = [key for key in identity if state[\'identity\'].get(key) != identity[key]]\n        raise ValueError(\'Resume rejected: changed \' + \', \'.join(changed))\n    epoch = state[\'completed_epochs\']\n    if epoch != len(state[\'history\']) or state[\'global_step\'] != epoch * identity[\'steps_per_epoch\']:\n        raise ValueError(\'Checkpoint epoch/history/step mismatch\')\n    if state[\'scheduler\'][\'last_epoch\'] != state[\'global_step\']:\n        raise ValueError(\'Checkpoint scheduler progress mismatch\')\n    return state\n\n\ndef save_epoch(run_dir, epoch, model, optimizer, scheduler, history, identity, best_epoch, best_f1):\n    run_dir = Path(run_dir)\n    if epoch != len(history):\n        raise ValueError(\'Save only after a complete epoch and its evaluations\')\n    path = run_dir / \'epochs\' / (\'epoch_%02d.pt\' % epoch)\n    if path.exists():\n        raise FileExistsError(\'Refusing to overwrite \' + str(path))\n    payload = dict(format=\'admul-epoch-v1\', completed_epochs=epoch,\n                   global_step=epoch * identity[\'steps_per_epoch\'], identity=identity,\n                   model=model.state_dict(), optimizer=optimizer.state_dict(),\n                   scheduler=scheduler.state_dict(), rng=rng_state(),\n                   best_epoch=best_epoch, best_f1=best_f1, history=history)\n    atomic_save(path, payload)\n    atomic_json(run_dir / \'history.json\', history)\n    atomic_json(run_dir / \'latest.json\', dict(completed_epochs=epoch, checkpoint=str(path.relative_to(run_dir)),\n                                            best_epoch=best_epoch, best_dev_f1=best_f1))\n    return path\n', 'smoke.py': '"""Exercise the real author model/train loop with tiny random DeBERTa on Kaggle."""\nimport tempfile\nfrom pathlib import Path\nfrom types import SimpleNamespace as NS\n\nimport torch\nimport torch.utils.checkpoint\nfrom transformers import DebertaV2Config, DebertaV2Model, AdamW, get_linear_schedule_with_warmup\nfrom torch.utils.data import DataLoader\n\nfrom model import DeBERTa_base\nfrom data_utils import dataset, collate_fn\nfrom train_val import train, set_random_seeds\nfrom utils import Lamda\n\n\ndef main():\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'GPU required for the Kaggle startup smoke test\')\n    set_random_seeds(4)\n    with tempfile.TemporaryDirectory(prefix=\'admul-smoke-\') as directory:\n        config = DebertaV2Config(vocab_size=32, hidden_size=24, num_hidden_layers=1,\n                                 num_attention_heads=4, intermediate_size=48,\n                                 max_position_embeddings=32, relative_attention=True,\n                                 pos_att_type=[\'p2c\', \'c2p\'], position_buckets=8,\n                                 max_relative_positions=16, conv_kernel_size=3,\n                                 norm_rel_ebd=\'layer_norm\')\n        DebertaV2Model(config).save_pretrained(directory)\n        args = NS(DATA=NS(plm=directory),\n                  MODEL=NS(dropout=0., embed_dim=24, num_classes=2, num_tasks=2),\n                  TRAIN=NS(wsd_weight=.2, global_weight=.1, local_weight=.1))\n        model = DeBERTa_base(args).cuda()\n        model.plm.gradient_checkpointing_enable()\n        optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=.01)\n        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=2)\n        examples = [[[1, 2, 3, 4, 5, 6, 7, 8], [0]*8, [1]*8, i % 2, i // 2] for i in range(4)]\n        loader = DataLoader(dataset(examples), batch_size=4, collate_fn=collate_fn)\n        losses = [torch.nn.CrossEntropyLoss(weight=torch.tensor(weight).cuda())\n                  for weight in ([1., 4.], [1., 1.], [1., 1.])]\n        train(0, model, *losses, optimizer, loader, 2, Lamda(.1, .8), args, scheduler=scheduler)\n        if not optimizer.state or not all(torch.isfinite(p).all().item() for p in model.parameters()):\n            raise RuntimeError(\'Tiny checkpointed training smoke test failed\')\n        model.eval()\n        batch = next(iter(loader))\n        with torch.no_grad():\n            logits = model(*(value.cuda() for value in batch[:3]), domain_idx=1)\n        if tuple(logits.shape) != (4, 2) or not torch.isfinite(logits).all().item():\n            raise RuntimeError(\'Tiny evaluation smoke test failed\')\n    torch.cuda.empty_cache()\n    print(\'SMOKE OK: author model, gradient checkpointing, backward, HF AdamW and evaluation\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n'}
SOURCE_COMMIT = '4847c72a4f8228a9197d1cbd0664eab7ddb0cfc8'
for name, content in SOURCES.items():
    destination = WORK / 'src' / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.read_text(encoding='utf-8') != content:
        raise ValueError(f'Code thay đổi giữa các phiên: {name}. Dùng cùng notebook để resume.')
    destination.write_text(content, encoding='utf-8')
marker = WORK / 'admul_run.json'
if not marker.exists():
    marker.write_text(json.dumps(dict(format='admul-epoch-v1', source_commit=SOURCE_COMMIT)), encoding='utf-8')
print('Author source commit:', SOURCE_COMMIT)


In [ ]:
# Môi trường cũ nằm ngoài /kaggle/working, không chiếm dung lượng Output.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.6.17'], check=True)
uv = [sys.executable, '-m', 'uv']
env_root = Path('/tmp/admul-paper-env')
subprocess.run(uv + ['venv', '--python', '3.10.14', '--seed', str(env_root)], check=True)
PYTHON = str(env_root / 'bin/python')
subprocess.run(uv + ['pip', 'install', '--python', PYTHON, '--no-cache',
    'torch==1.13.1+cu116', '--extra-index-url', 'https://download.pytorch.org/whl/cu116',
    '--index-strategy', 'unsafe-best-match',
    'transformers==4.18.0', 'numpy==1.21.3', 'tokenizers==0.12.1',
    'huggingface-hub==0.10.1', 'sentencepiece==0.1.99', 'protobuf==3.20.3', 'six==1.16.0',
    'scikit-learn==1.2.2', 'scipy==1.10.1', 'yacs==0.1.8', 'tqdm==4.66.5',
    'filelock==3.13.1', 'packaging==23.2', 'pyyaml==6.0.1', 'regex==2023.12.25',
    'requests==2.31.0', 'sacremoses==0.1.1', 'typing-extensions==4.9.0',
    'joblib==1.3.2', 'threadpoolctl==3.2.0', 'click==8.1.7',
    'certifi==2024.2.2', 'charset-normalizer==3.3.2', 'idna==3.6', 'urllib3==2.2.1'], check=True)
child_env = os.environ.copy()
child_env.update(CUDA_VISIBLE_DEVICES='0', CUBLAS_WORKSPACE_CONFIG=':4096:8',
                 PYTHONHASHSEED=str(SEED), TOKENIZERS_PARALLELISM='false',
                 USE_TF='0', USE_FLAX='0', HF_HOME='/tmp/admul-hf')
subprocess.run([PYTHON, '-c',
    'import torch,transformers,numpy; print(torch.__version__,transformers.__version__,numpy.__version__); '
    'import six; print("six ready:", six.__version__); '
    'import torch.utils.checkpoint; '
    'from transformers.models.deberta_v2.tokenization_deberta_v2 import DebertaV2Tokenizer; '
    'assert torch.cuda.is_available(), "CUDA không hoạt động trong môi trường riêng"; '
    'print(torch.cuda.get_device_name(0))'], env=child_env, check=True)
(WORK / 'run').mkdir(exist_ok=True)
lock_path = WORK / 'run/requirements.lock.txt'
new_lock = subprocess.check_output([PYTHON, '-m', 'pip', 'freeze'], text=True)
if lock_path.exists() and lock_path.read_text() != new_lock:
    raise RuntimeError('Dependency lock changed. Do not resume with different dependencies.')
lock_path.write_text(new_lock)
subprocess.run([PYTHON, '-u', str(WORK / 'src/smoke.py')], cwd=WORK, env=child_env, check=True)


In [ ]:
# Snapshot bất biến; không dùng model đã fine-tune từ Input cũ.
import urllib.request
MODEL_ID = 'microsoft/deberta-v3-base'
REVISION = '8ccc9b6f36199bec6961081d44eb72fb3f7353f3'
backbone = WORK / 'deberta-v3-base'
backbone.mkdir(exist_ok=True)
for filename in ['config.json', 'pytorch_model.bin', 'spm.model', 'tokenizer_config.json']:
    destination = backbone / filename
    if not destination.exists():
        print('Downloading pretrained:', filename, flush=True)
        request = urllib.request.Request(f'https://huggingface.co/{MODEL_ID}/resolve/{REVISION}/{filename}',
                                         headers={'User-Agent': 'AdMul-reproduction'})
        temporary = destination.with_suffix(destination.suffix + '.part')
        with urllib.request.urlopen(request, timeout=120) as response, temporary.open('wb') as output:
            shutil.copyfileobj(response, output, length=8*1024*1024)
        os.replace(temporary, destination)
(backbone / 'source_revision.txt').write_text(f'{MODEL_ID}\n{REVISION}\n', encoding='utf-8')


In [ ]:
remaining = SESSION_HOURS * 3600 - (time.monotonic() - SESSION_STARTED)
if remaining <= 0:
    raise RuntimeError('Hết ngân sách phiên trong bước chuẩn bị. Chạy lại trong phiên mới.')
command = [PYTHON, '-u', str(WORK / 'src/runner.py'), '--work', str(WORK),
           '--seed', str(SEED), '--max-session-epochs', str(MAX_SESSION_EPOCHS),
           '--seconds-left', str(remaining)]
with (WORK / 'run/console.log').open('a', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=WORK, env=child_env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        returncode = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
if returncode:
    raise RuntimeError(f'Runner lỗi {returncode}; xem console.log. Không tự giảm batch hay đổi cấu hình.')


In [ ]:
result = json.loads((WORK / 'run/result.json').read_text())
print(json.dumps(result, indent=2))
print('Checkpoint epochs:', [p.name for p in sorted((WORK / 'run/epochs').glob('epoch_*.pt'))])
if result['training_complete']:
    print('Đã xong 8 epoch. Model tốt nhất:', result['best_checkpoint'])
else:
    print('Phiên đã dừng. Gắn Output phiên này vào phiên mới để chạy epoch', result['next_epoch'])
print('Giữ toàn bộ thư mục AdMulPaper trong Output, không chỉ một file .pt.')
